# TFT runner for Colab

This notebook clones the repo, installs dependencies, checks GPU, and trains any of these models:

- `tft_baseline`
- `mlp_features`
- `no_attention`
- `no_lstm`
- `transformer_only`


## 1. Clone repo from git


In [ ]:
!git clone https://github.com/HannaVallner/tft_electricity.git
%cd tft_electricity


## 2. Install requirements


In [ ]:
!pip install -r requirements.txt

## 3. Check GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Mount Google Drive for checkpoints/results


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. Replacement training block


In [ ]:
print(r"""from pathlib import Path
import argparse
import importlib
import time

import pandas as pd
import torch
from torch.utils.data import DataLoader

from data_formatter import ElectricityFormatter
from dataset import TFTDataset


MODEL_REGISTRY = {
    "tft_baseline": "models.tft_baseline",
    "mlp_features": "models.mlp_features",
    "no_attention": "models.no_attention",
    "no_lstm": "models.no_lstm",
    "transformer_only": "models.transformer_only",
}


def load_model_class_and_loss(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(
            f"Unknown model '{model_name}'. Choose from: {list(MODEL_REGISTRY.keys())}"
        )

    module = importlib.import_module(MODEL_REGISTRY[model_name])

    if not hasattr(module, "TemporalFusionTransformer"):
        raise ValueError(f"{MODEL_REGISTRY[model_name]} is missing TemporalFusionTransformer")

    if not hasattr(module, "quantile_loss"):
        raise ValueError(f"{MODEL_REGISTRY[model_name]} is missing quantile_loss")

    return module.TemporalFusionTransformer, module.quantile_loss


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            inputs = batch["inputs"].to(device)
            targets = batch["outputs"].to(device)

            predictions = model(inputs)
            loss = loss_fn(targets, predictions)

            total_loss += loss.item()
            num_batches += 1

    if num_batches == 0:
        raise ValueError("Validation dataloader produced zero batches.")

    return total_loss / num_batches


def train_one_epoch(model, dataloader, optimizer, loss_fn, device, epoch_index, print_every):
    model.train()
    total_loss = 0.0
    num_batches = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(dataloader, start=1):
        inputs = batch["inputs"].to(device)
        targets = batch["outputs"].to(device)

        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(targets, predictions)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if batch_idx % print_every == 0:
            elapsed = time.time() - start_time
            avg_loss = total_loss / num_batches
            print(
                f"Epoch {epoch_index + 1} | Batch {batch_idx}/{len(dataloader)} | "
                f"Train Loss: {avg_loss:.6f} | Elapsed: {elapsed:.1f}s"
            )

    if num_batches == 0:
        raise ValueError("Training dataloader produced zero batches.")

    return total_loss / num_batches


def select_subset_of_ids(df, num_ids, random_state=42):
    unique_ids = df["id"].drop_duplicates()

    if num_ids > len(unique_ids):
        raise ValueError(
            f"Requested {num_ids} ids, but only {len(unique_ids)} are available."
        )

    selected_ids = unique_ids.sample(n=num_ids, random_state=random_state)
    subset = df[df["id"].isin(selected_ids)].copy()
    subset = subset.sort_values(["id", "hours_from_start"]).reset_index(drop=True)
    return subset


def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_class, loss_fn = load_model_class_and_loss(args.model)

    data_path = Path(args.data_path)
    save_dir = Path(args.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model_save_path = save_dir / f"{args.model}_best.pt"

    print("Loading processed electricity data...")
    df = pd.read_csv(data_path)
    print(f"Full data shape: {df.shape}")

    print("Formatting data...")
    formatter = ElectricityFormatter()
    train_df, valid_df, test_df = formatter.split_data(df)

    print("Full train dataframe shape:", train_df.shape)
    print("Full valid dataframe shape:", valid_df.shape)
    print("Full test dataframe shape:", test_df.shape)

    if args.num_train_ids is not None:
        train_df = select_subset_of_ids(train_df, args.num_train_ids, random_state=42)
    if args.num_valid_ids is not None:
        valid_df = select_subset_of_ids(valid_df, args.num_valid_ids, random_state=42)

    print("Train dataframe shape:", train_df.shape)
    print("Valid dataframe shape:", valid_df.shape)

    train_dataset = TFTDataset(train_df, formatter)
    valid_dataset = TFTDataset(valid_df, formatter)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False)

    print(f"Using device: {device}")
    model = model_class(formatter).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)

    best_valid_loss = float("inf")

    for epoch in range(args.num_epochs):
        epoch_start = time.time()

        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            epoch_index=epoch,
            print_every=args.print_every,
        )

        valid_loss = evaluate(model=model, dataloader=valid_loader, loss_fn=loss_fn, device=device)
        epoch_time = time.time() - epoch_start

        print(
            f"\nEpoch {epoch + 1}/{args.num_epochs} completed | "
            f"Train Loss: {train_loss:.6f} | Valid Loss: {valid_loss:.6f} | Time: {epoch_time:.1f}s\n"
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), model_save_path)
            print(f"New best model saved to: {model_save_path}")

    print("Training finished.")
    print(f"Best validation loss: {best_valid_loss:.6f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True, choices=list(MODEL_REGISTRY.keys()))
    parser.add_argument("--data_path", type=str, default="data/electricity_processed.csv")
    parser.add_argument("--save_dir", type=str, default="checkpoints")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1e-3)
    parser.add_argument("--num_epochs", type=int, default=2)
    parser.add_argument("--print_every", type=int, default=20)
    parser.add_argument("--num_train_ids", type=int, default=None)
    parser.add_argument("--num_valid_ids", type=int, default=None)
    args = parser.parse_args()
    main(args)
""")


## 6. Small debug run


In [ ]:
!python train.py     --model baseline     --data_path data/electricity_processed.csv     --save_dir checkpoints     --batch_size 32     --learning_rate 1e-3     --num_epochs 2     --num_train_ids 20     --num_valid_ids 20


## 7. Full training commands


In [ ]:
# baseline
!python train.py --model tft_baseline --data_path data/electricity_processed.csv --save_dir checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30

# mlp_features
# !python train.py --model mlp_features --data_path data/electricity_processed.csv --save_dir checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30

# no_attention
# !python train.py --model no_attention --data_path data/electricity_processed.csv --save_dir checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30

# no_lstm
# !python train.py --model no_lstm --data_path data/electricity_processed.csv --save_dir checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30

# transformer_only
# !python train.py --model transformer_only --data_path data/electricity_processed.csv --save_dir checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30


## 8. Save checkpoints to Google Drive instead of Colab temp storage


In [ ]:
# Example:
# !python train.py --model tft_baseline --data_path data/electricity_processed.csv --save_dir /content/drive/MyDrive/tft_checkpoints --batch_size 64 --learning_rate 1e-3 --num_epochs 30
